In [1]:
#0 Load Libraries and Configurations

import os
import wrds
import pandas as pd
import numpy as np

# ---------- User-configurable paths ----------
PATH_DATA_INTERMEDIATE = "/Users/nglei/Desktop/Academics/SMU/Modules/QF600 Asset Pricing/Project/Project Code/cz_data/intermediate"  # <-- change this
os.makedirs(PATH_DATA_INTERMEDIATE, exist_ok=True)

OUT_PARQUET = os.path.join(PATH_DATA_INTERMEDIATE, "IBES_UnadjustedActuals.parquet")
OUT_CSV     = os.path.join(PATH_DATA_INTERMEDIATE, "IBES_UnadjustedActuals.csv")

In [ ]:
#1 Load IBES Data

SQL = """
SELECT
    a.*
FROM ibes.actpsumu_epsus AS a
WHERE a.measure = 'EPS'
  AND a.statpers >= DATE '2000-01-01'
;
"""

In [ ]:
#2 IBES Data Extraction From WRDS

db = wrds.Connection()
df = db.raw_sql(SQL, date_cols=["statpers", "fy0edats"])

In [ ]:
#3 Data Cleaning
if "shout" in df.columns:
    df = df.rename(columns={"shout": "shoutIBESUnadj"})

# ---------------- Monthly time index ----------------
df["time_avail_m"] = df["statpers"].dt.to_period("M").dt.to_timestamp("MS")

# ---------------- Keep first obs each month (per ticker) ----------------
# Stata: egen id=group(ticker); bys id time_av: keep if _n==1 (earliest statpers)
df = df.sort_values(["ticker", "time_avail_m", "statpers"], kind="mergesort")
df = df.drop_duplicates(subset=["ticker", "time_avail_m"], keep="first")

# ---------------- tsfill monthly + forward-fill within ticker ----------------
# Build complete monthly index per ticker and forward-fill selected cols
ffill_cols = [c for c in ["int0a", "fy0a", "shoutIBESUnadj", "ticker"] if c in df.columns]

out_frames = []
for tick, g in df.groupby("ticker", sort=False):
    g = g.set_index("time_avail_m").sort_index()
    # Complete monthly span for this ticker
    full_idx = pd.date_range(g.index.min(), g.index.max(), freq="MS")
    g = g.reindex(full_idx)
    # Forward-fill selected columns within ticker
    if ffill_cols:
        g[ffill_cols] = g[ffill_cols].ffill()
    # Restore index name/column
    g.index.name = "time_avail_m"
    g = g.reset_index()
    out_frames.append(g)

panel = pd.concat(out_frames, ignore_index=True)

# ---------------- Drop statpers, prep for merge, rename ticker ----------------
if "statpers" in panel.columns:
    panel = panel.drop(columns=["statpers"])
panel = panel.rename(columns={"ticker": "tickerIBES"})

# ---------------- Save ----------------
panel.to_parquet(OUT_PARQUET, index=False)
panel.to_csv(OUT_CSV, index=False)

print("Saved:")
print(" -", OUT_PARQUET)
print(" -", OUT_CSV)
print(panel.head())
if "shout" in df.columns:
    df = df.rename(columns={"shout": "shoutIBESUnadj"})

# ---------------- Monthly time index ----------------
df["time_avail_m"] = df["statpers"].dt.to_period("M").dt.to_timestamp("MS")

# ---------------- Keep first obs each month (per ticker) ----------------
# Stata: egen id=group(ticker); bys id time_av: keep if _n==1 (earliest statpers)
df = df.sort_values(["ticker", "time_avail_m", "statpers"], kind="mergesort")
df = df.drop_duplicates(subset=["ticker", "time_avail_m"], keep="first")

# ---------------- tsfill monthly + forward-fill within ticker ----------------
# Build complete monthly index per ticker and forward-fill selected cols
ffill_cols = [c for c in ["int0a", "fy0a", "shoutIBESUnadj", "ticker"] if c in df.columns]

out_frames = []
for tick, g in df.groupby("ticker", sort=False):
    g = g.set_index("time_avail_m").sort_index()
    # Complete monthly span for this ticker
    full_idx = pd.date_range(g.index.min(), g.index.max(), freq="MS")
    g = g.reindex(full_idx)
    # Forward-fill selected columns within ticker
    if ffill_cols:
        g[ffill_cols] = g[ffill_cols].ffill()
    # Restore index name/column
    g.index.name = "time_avail_m"
    g = g.reset_index()
    out_frames.append(g)

panel = pd.concat(out_frames, ignore_index=True)

# ---------------- Drop statpers, prep for merge, rename ticker ----------------
if "statpers" in panel.columns:
    panel = panel.drop(columns=["statpers"])
panel = panel.rename(columns={"ticker": "tickerIBES"})

# ---------------- Save ----------------
panel.to_parquet(OUT_PARQUET, index=False)
panel.to_csv(OUT_CSV, index=False)

print("Saved:")
print(" -", OUT_PARQUET)
print(" -", OUT_CSV)
print(panel.head())